In [19]:
# Since gensim's Word2Vec already handles the architecture internally, we'll do two things: (1) train a CBOW model explicitly, and (2) demonstrate the actual "fill in the blank" prediction mechanism directly, so you can see CBOW doing what the theory described

In [20]:
from gensim.models import Word2Vec

In [21]:
# A slightly bigger toy corpus with "interest" appearing in a consistent context
sentences = [
    ["the", "fed", "raised", "interest", "rates", "sharply"],
    ["the", "central", "bank", "raised", "interest", "rates", "today"],
    ["investors", "worried", "about", "rising", "interest", "rates"],
    ["the", "fed", "cut", "interest", "rates", "last", "quarter"],
    ["analysts", "expect", "interest", "rates", "to", "stay", "high"],
    ["the", "stock", "market", "crashed", "after", "the", "announcement"],
    ["the", "market", "rallied", "despite", "the", "news", "today"]
]

In [22]:
# Train with CBOW explicitly (sg=0)
cbow_model = Word2Vec(
    sentences,
    window=2,  # Maximum distance b/w the current and predicted word within a sentence
    vector_size=50,
    min_count=1,
    sg=0    # sg = 0 ==> CBOW, sg=1 for skip-gram
)

In [23]:
# 1 - Check the vector exists and has the right shape. This confirms the word got a proper 50-dimensional vector - same as before. This is the end product of training, stored in wv lookup table
print(f"Vector shape for 'interest': {cbow_model.wv["interest"].shape}")

Vector shape for 'interest': (50,)


In [24]:
# 2 - Most similar words to "interest"
# Finds words whose learned vectors are closest to "interest" in the vector space. On this small corpus, "analysts" scoring highest makes less sense, but on real data, it will be more clear
print(f"Most similar to 'interest': {cbow_model.wv.most_similar("interest", topn=3)}")

Most similar to 'interest': [('analysts', 0.19688619673252106), ('today', 0.16540955007076263), ('after', 0.15550071001052856)]


In [28]:
# 3 - Directly demonstrate the "fill in the blank" mechanism
# This is the demonstration of CBOW. You feed in a list of context words and ask the trained model "what word do you think belongs in the middle of this context?" It returns a ranked list of candidate words with prob scores.
context_words = ["fed", "raised", "rates", "sharply"]
predicted = cbow_model.predict_output_word(context_words, topn=5)

In [29]:
predicted

[('analysts', np.float32(0.034483846)),
 ('quarter', np.float32(0.03448359)),
 ('interest', np.float32(0.03448345)),
 ('cut', np.float32(0.03448339)),
 ('announcement', np.float32(0.034483314))]

In [30]:
print(f"Given Context: {context_words}, model predicts:")
for word, prob, in predicted:
    print(f"{word}: {prob:.7f}")

Given Context: ['fed', 'raised', 'rates', 'sharply'], model predicts:
analysts: 0.0344838
quarter: 0.0344836
interest: 0.0344835
cut: 0.0344834
announcement: 0.0344833
